# Description

This notebook is used for a one time task of finding emails and profiles of target VC investors

In [2]:
#!sudo /bin/bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

     |████████████████████████████████| 12.0 MB 9.7 MB/s eta 0:00:01
     |████████████████████████████████| 139 kB 79.5 MB/s eta 0:00:01
     |████████████████████████████████| 96 kB 6.4 MB/s  eta 0:00:01
     |████████████████████████████████| 220 kB 81.4 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 88.1 MB/s eta 0:00:01
     |████████████████████████████████| 50 kB 11.1 MB/s eta 0:00:01


In [5]:
import logging
import os

import numpy as np
import pandas as pd

import ck_marketing.dropcontact.dropcontact_api as cmdrdrap
import ck_marketing.hunterio.hunter_api as cmhuhuap
import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint
from ck_marketing.hunterio.hunter_api import GoogleSheetsHelper, HunterIO

In [6]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Print system signature.
_LOG.info("%s", henv.get_system_signature()[0])

# Configure the notebook style.
hprint.config_notebook()

INFO  > cmd='/venv/lib/python3.9/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-9e9a6692-c133-4c97-a0c0-f036f1be6bbe.json'
INFO  # Git
  branch_name='CmTask9143_Build_target_VC_investor_lists'
  hash='c3fd664b5'
  # Last commits:
    * c3fd664b5 Sonaal   checkpoint (#9137)                                                (  14 hours ago) Mon Jul 22 22:44:19 2024  (HEAD -> CmTask9143_Build_target_VC_investor_lists, origin/master, origin/HEAD, origin/CmampTask8740_Convert_main.sh_Script_to_Python, master)
    * 2ce06d939 GP Saggese Rename files                                                      (  17 hours ago) Mon Jul 22 19:54:44 2024           
    * e25e7124f Vedanshu Joshi CmTask9089 Unit test Airflow utils function get_flatten_account_cmd (#9105) (  17 hours ago) Mon Jul 22 19:30:34 2024           
# Machine info
  system=Linux
  node name=f59e38dbbf8d
  release=5.15.0-1056-aws
  version=#61~20.04.1-Ubuntu SMP Wed Mar 13 17:40:41 UTC 2024
  machine=

In [20]:
hunter_api_key = os.getenv("Hunter_API_KEY")
dropcontact_api_key = os.getenv("Drop_API_KEY")
phantom_api_key = os.getenv("Phantom_API_KEY")

In [9]:
google_creds_path = "service.json"
google_sheet_helper = GoogleSheetsHelper(google_creds_path)

In [10]:
file_id = "1mkBjvGDjlwXe-h1OOZjCGzcU2iddkj8rZlCF-AAuyRY"

In [11]:
df = google_sheet_helper.read_sheet(file_id)

In [12]:
df["firstName"] = df["Name"].str.split().str[0]
df["lastName"] = df["Name"].str.split().str[-1]

In [15]:
sheet = google_sheet_helper.google_account.open_by_key(file_id)
tab_name = "cleaned_profiles_1"
cleaned_profiles_tab = sheet.add_worksheet(title=tab_name, rows="100", cols="20")
google_sheet_helper.write_results(file_id, df, tab_name)

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(


INFO  Email extraction completed. Results saved in the new tab: cleaned_profiles_1


# HunterIO - Extract emails

In [ ]:
first_name_col = "firstName"
last_name_col = "lastName"
company_col = "Firm"

cmhuhuap.process_records(
    api_key=hunter_api_key,
    google_creds_path=google_creds_path,
    file_id=file_id,
    first_name_col=first_name_col,
    last_name_col=last_name_col,
    company_col=company_col,
    tab_name=tab_name,
)

# DropContact - Extract Remaining emails

In [17]:
df_drop = google_sheet_helper.read_sheet(file_id, "hunter_results")

In [18]:
missing_email_df = df_drop[
    df_drop["hunter_extracted_email"].isna()
    | (df_drop["hunter_extracted_email"] == "")
]

In [21]:
# Prepare data for DropContact
first_names = missing_email_df["firstName"].tolist()
last_names = missing_email_df["lastName"].tolist()
company_names = missing_email_df["Firm"].tolist()

# Get emails from DropContact
dropcontact_results_df = cmdrdrap.get_email_from_dropcontact(
    first_names, last_names, company_names, dropcontact_api_key
)

Processing batches:   0%|                                                     | 0/1 [00:00<?, ?it/s]

Starting query batch 0.
Batch 0: Query ID: xvcxzzskszwwimo.


Processing batches: 100%|############################################| 1/1 [01:45<00:00, 105.77s/it]

Batch 0: Query finished. Credits left: 790.
Batch 0 completed in 105.77 seconds.
Total processing time: 105.77 seconds.


In [22]:
# Replace special float values with NaN
df_drop.replace([np.inf, -np.inf], np.nan, inplace=True)
dropcontact_results_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Rename columns in dropcontact_results_df to match df_drop
dropcontact_results_df.rename(
    columns={"first name": "firstName", "last name": "lastName"}, inplace=True
)

# Merge the dataframes on 'firstNames' and 'lastName', keeping only the 'email' column from dropcontact_results_df
merged_df = pd.merge(
    df_drop,
    dropcontact_results_df[["firstName", "lastName", "email"]],
    on=["firstName", "lastName"],
    how="left",
)

# Rename the 'email' column to 'dropcontact_mail'
merged_df.rename(columns={"email": "dropcontact_mail"}, inplace=True)

# Count the non-null values in the 'dropcontact_mail' column
email_count = merged_df[
    merged_df["dropcontact_mail"].notna() & (merged_df["dropcontact_mail"] != "")
]["dropcontact_mail"].count()
print(f"Number of emails found: {email_count}")
print(f"Number of profile emails hunter could not find: {len(missing_email_df)}")

Number of emails found: 19
Number of profile emails hunter could not find: 37


In [23]:
merged_df.replace({np.nan: "", np.inf: "", -np.inf: ""}, inplace=True)
cleaned_profiles_tab = sheet.add_worksheet(
    title="hunter_drop_emails", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, merged_df, "hunter_drop_emails")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(


INFO  Email extraction completed. Results saved in the new tab: hunter_drop_emails


In [24]:
merged_df["all_emails"] = (
    merged_df["hunter_extracted_email"]
    .fillna("")
    .replace("", pd.NA)
    .combine_first(merged_df["dropcontact_mail"])
)
merged_df.replace({np.nan: "", np.inf: "", -np.inf: ""}, inplace=True)

In [25]:
cleaned_profiles_tab = sheet.add_worksheet(
    title="all_emails", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, merged_df, "all_emails")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(


INFO  Email extraction completed. Results saved in the new tab: all_emails


# HunterIO - Verify emails

In [26]:
hunter_instance = HunterIO(hunter_api_key)
verified_df = hunter_instance.verify_emails(merged_df, "all_emails")

WARNING Error verifying email : 400 Client Error: Bad Request for url: https://api.hunter.io/v2/email-verifier?email=&api_key=89afbf8e860cc8a9af6d8c66d0bfd0844815af16
WARNING Error verifying email : 400 Client Error: Bad Request for url: https://api.hunter.io/v2/email-verifier?email=&api_key=89afbf8e860cc8a9af6d8c66d0bfd0844815af16
WARNING Error verifying email : 400 Client Error: Bad Request for url: https://api.hunter.io/v2/email-verifier?email=&api_key=89afbf8e860cc8a9af6d8c66d0bfd0844815af16
WARNING Error verifying email : 502 Server Error: Bad Gateway for url: https://api.hunter.io/v2/email-verifier?email=&api_key=89afbf8e860cc8a9af6d8c66d0bfd0844815af16
WARNING Error verifying email : 400 Client Error: Bad Request for url: https://api.hunter.io/v2/email-verifier?email=&api_key=89afbf8e860cc8a9af6d8c66d0bfd0844815af16
WARNING Error verifying email : 400 Client Error: Bad Request for url: https://api.hunter.io/v2/email-verifier?email=&api_key=89afbf8e860cc8a9af6d8c66d0bfd0844815af1

In [27]:
cleaned_profiles_tab = sheet.add_worksheet(
    title="hunter_verification", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, verified_df, "hunter_verification")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(


INFO  Email extraction completed. Results saved in the new tab: hunter_verification


# Final sheet

In [28]:
final_df = verified_df
# Step 2: Filter out rows where 'hunter_extracted_email' is empty.
final_df = final_df[
    final_df["all_emails"].notna() & (final_df["all_emails"] != "")
]

cleaned_profiles_tab = sheet.add_worksheet(
    title="found_and_verified_mails", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, final_df, "found_and_verified_mails")

INFO  Email extraction completed. Results saved in the new tab: found_and_verified_mails
